In [6]:
import torch
from torch import nn
import torch.nn.functional as F

In [7]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 6, 3)
        self.conv2 = nn.Conv2d(6, 16, 3)
        self.fc1 = nn.Linear(16 * 6 * 6, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

    def forward(self, x):
        x =self.conv1(x)
        x = F.max_pool2d(x, 2)
        x = torch.relu(x)

        x = self.conv2(x)
        x = F.max_pool2d(x, 2)
        x = torch.relu(x)

        x = x.view(-1, self.num_flat_features(x))   
        x = self.fc1(x)
        x = torch.relu(x)

        x = self.fc2(x)
        x = torch.relu(x)

        x = self.fc3(x)
        return x
    

In [8]:
Net()

Net(
  (conv1): Conv2d(1, 6, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=576, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [ ]:
# 1, 1, 28, 28: incorrect input size for conv1, which expects 32x32 images
input_image = torch.randn(1, 1, 32, 32)
net = Net()
output = net(input_image)
output.shape

torch.Size([1, 10])

# 📚 Bài học về `Conv2d`, `Linear` và `view()` trong PyTorch

---

# 1. `nn.Conv2d`

```python
# nn.Conv2d(in_channels, out_channels, kernel_size)

self.conv1 = nn.Conv2d(1, 6, 3)
```

## Ý nghĩa các tham số

| Tham số | Giá trị | Ý nghĩa |
|---------|---------|----------|
| `in_channels` | `1` | Số channel của ảnh đầu vào. <br>• Ảnh Grayscale: `1` channel <br>• Ảnh RGB: `3` channels |
| `out_channels` | `6` | Số feature maps đầu ra. Điều này cũng có nghĩa Conv sẽ học **6 kernel (filters)** khác nhau. |
| `kernel_size` | `3` | Kích thước kernel là **3×3**. |

### Tham số mặc định

- `stride = 1`
  - Kernel dịch chuyển **1 pixel** sau mỗi lần quét.

- `padding = 0`
  - Không thêm viền `0` quanh ảnh.

---

## Kernel và Feature Map

> **Kernel (Filter)** là ma trận trọng số được **học** trong quá trình huấn luyện.

> **Feature Map** là **kết quả đầu ra** sau khi kernel quét toàn bộ ảnh.

Ví dụ:

```text
Kernel 1
    ↓
Phát hiện cạnh ngang
    ↓
Feature Map 1

Kernel 2
    ↓
Phát hiện cạnh dọc
    ↓
Feature Map 2

...
```

---

# 2. `nn.Linear`

```python
self.fc1 = nn.Linear(16 * 6 * 6, 120)
```

## Tại sao là `16 × 6 × 6`?

Đây **không phải là một hằng số**, mà phụ thuộc vào:

- Kích thước ảnh đầu vào
- Các lớp Convolution
- Các lớp Pooling

Trong ví dụ này, ảnh đầu vào có kích thước:

```text
1 × 32 × 32
```

### Quá trình biến đổi kích thước

```text
Input
1 × 32 × 32
        │
        ▼
Conv1 (3×3)
        │
        ▼
6 × 30 × 30
        │
        ▼
MaxPool (2×2)
        │
        ▼
6 × 15 × 15
        │
        ▼
Conv2 (3×3)
        │
        ▼
16 × 13 × 13
        │
        ▼
MaxPool (2×2)
        │
        ▼
16 × 6 × 6
```

### Tính toán

Conv1:

```text
(32 - 3) / 1 + 1 = 30
```

Pool:

```text
30 / 2 = 15
```

Conv2:

```text
(15 - 3) / 1 + 1 = 13
```

Pool:

```text
13 / 2 = 6 (lấy phần nguyên)
```

Output cuối cùng:

```text
16 × 6 × 6
```

Trong đó:

- `16` là số feature maps (output channels của Conv2)
- `6 × 6` là kích thước của mỗi feature map

Tổng số phần tử:

```text
16 × 6 × 6 = 576
```

Do đó:

```python
nn.Linear(576, 120)
```

hoặc

```python
nn.Linear(16 * 6 * 6, 120)
```

là hoàn toàn giống nhau.

> ⚠️ Nếu thay đổi kích thước ảnh đầu vào (ví dụ 28×28 hoặc 64×64) thì giá trị `16 × 6 × 6` cũng sẽ thay đổi.

---

# 3. `view()`

```python
x = x.view(-1, self.num_flat_features(x))
```

## Tại sao cần `view()`?

Sau các lớp Convolution và Pooling, dữ liệu có dạng:

```text
(batch, 16, 6, 6)
```

Trong khi đó `nn.Linear` chỉ nhận dữ liệu dạng:

```text
(batch, số_features)
```

Vì vậy cần **Flatten** tensor.

```text
(batch, 16, 6, 6)

        │
        ▼

(batch, 576)
```

`view()` chỉ làm nhiệm vụ:

- Thay đổi **shape** của tensor.
- Không thay đổi giá trị dữ liệu.
- Không học thêm bất kỳ đặc trưng nào.

---

## Ý nghĩa của `-1`

```python
x.view(-1, 576)
```

`-1` nghĩa là:

> **PyTorch sẽ tự tính chiều Batch cho chúng ta.**

Ví dụ:

```text
(32,16,6,6)
```

↓

```text
(32,576)
```

Hoặc

```text
(8,16,6,6)
```

↓

```text
(8,576)
```

---

# Tóm tắt toàn bộ Pipeline

```text
Input Image
      │
      ▼
Conv1
      │
      ▼
Feature Maps
      │
      ▼
MaxPool
      │
      ▼
Conv2
      │
      ▼
Feature Maps
      │
      ▼
MaxPool
      │
      ▼
Tensor (16 × 6 × 6)
      │
      ▼
Flatten (view)
      │
      ▼
Vector (576)
      │
      ▼
Linear (576 → 120)
      │
      ▼
Linear
      │
      ▼
Output
```